# Importation des données 

In [109]:
from sklearn.datasets import fetch_openml

freq = fetch_openml(data_id=41214, as_frame=True).frame
sev = fetch_openml(data_id=41215, as_frame=True).frame

print("La première table freq contient les données de " + str(freq.shape[0]) + " polices d'assurance et " + str(freq.shape[1]) + " variables.")
print("La deuxième table sev contient les données de " + str(sev.shape[0]) + " sinistres et " + str(sev.shape[1]) + " variables.")
print(freq.head())
print(sev.head())


La première table freq contient les données de 678013 polices d'assurance et 12 variables.
La deuxième table sev contient les données de 26639 sinistres et 2 variables.
   IDpol  ClaimNb  Exposure Area  VehPower  VehAge  DrivAge  BonusMalus  \
0    1.0        1      0.10    D         5       0       55          50   
1    3.0        1      0.77    D         5       0       55          50   
2    5.0        1      0.75    B         6       2       52          50   
3   10.0        1      0.09    B         7       0       46          50   
4   11.0        1      0.84    B         7       0       46          50   

  VehBrand     VehGas  Density Region  
0      B12  'Regular'     1217    R82  
1      B12  'Regular'     1217    R82  
2      B12   'Diesel'       54    R22  
3      B12   'Diesel'       76    R72  
4      B12   'Diesel'       76    R72  
     IDpol  ClaimAmount
0     1552       995.20
1  1010996      1128.12
2  4024277      1851.11
3  4007252      1204.00
4  4046424      1204

## Nettoyage des données

Avant de commencer a manipuler les données, il est important de constater s'il y a des erreurs dans les tables de données. Par exemple nous allons regarder s'il y a des doublons dans les polices, des temps d'exposition qui depassent 1 an, ou s'il y a des fréquences de sinistres anormalement haute, ce qui traduirait des erreurs de saisies. (Fortement probable sur un jeu de données de cette taille)

In [ ]:
print("le nombre de polices dupliquées est de", freq['IDpol'].duplicated().sum())

le nombre de polices dupliquées est de 0


In [93]:
# verirification d'eventuelles valeurs manquantes
print(freq.isnull().sum())

IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64


In [94]:
# verification de l'intégrité des données
print(freq[['VehPower','VehAge','DrivAge','BonusMalus','Density']].describe())
print("nombre de score bonus/malus en dehors de la legislation : ", (freq['BonusMalus'] > 350).sum(), (freq['BonusMalus'] < 50).sum())
print("nombre de conducteurs avec un âge inférieur à 18 ans : ", (freq['DrivAge'] < 18).sum())

            VehPower         VehAge        DrivAge     BonusMalus  \
count  678013.000000  678013.000000  678013.000000  678013.000000   
mean        6.454631       7.044265      45.499122      59.761502   
std         2.050906       5.666232      14.137444      15.636658   
min         4.000000       0.000000      18.000000      50.000000   
25%         5.000000       2.000000      34.000000      50.000000   
50%         6.000000       6.000000      44.000000      50.000000   
75%         7.000000      11.000000      55.000000      64.000000   
max        15.000000     100.000000     100.000000     230.000000   

             Density  
count  678013.000000  
mean     1792.422405  
std      3958.646564  
min         1.000000  
25%        92.000000  
50%       393.000000  
75%      1658.000000  
max     27000.000000  
nombre de score bonus/malus en dehors de la legislation :  0 0
nombre de conducteurs avec un âge inférieur à 18 ans :  0


Aucune valeur anormale sur ces features, aucun ajustement n'est donc nécessaire.

In [95]:
print(freq['Exposure'].describe())
print((freq['Exposure'] > 1).sum())

count    678013.000000
mean          0.528750
std           0.364442
min           0.002732
25%           0.180000
50%           0.490000
75%           0.990000
max           2.010000
Name: Exposure, dtype: float64
1224


In [96]:
print(freq['Exposure'].describe())
print("Nombre de polices dont l'exposition est supérieur à 1 : ", (freq['Exposure'] > 1).sum())

count    678013.000000
mean          0.528750
std           0.364442
min           0.002732
25%           0.180000
50%           0.490000
75%           0.990000
max           2.010000
Name: Exposure, dtype: float64
Nombre de polices dont l'exposition est supérieur à 1 :  1224


Le fait que certaines exposition dépasse un an (le cas pour 1224 contrats) est une anomalie car on est sur un jeu de données qui est censé contenir des contrats annuels.

On ramène les temps d'exposition des contrats dépassant une durée de 1 an à 1 car il s'agit surement là d'une anomalie et de toute façon leur impact sera négligeable. Seulement 1224 sur 678 000 ...

In [ ]:

freq['Exposure'] = freq['Exposure'].clip(upper=1)

In [99]:
print(sev['ClaimAmount'].describe())

count    2.663900e+04
mean     2.278536e+03
std      2.929748e+04
min      1.000000e+00
25%      6.868100e+02
50%      1.172000e+03
75%      1.228080e+03
max      4.075401e+06
Name: ClaimAmount, dtype: float64


Un montant de 1 euro a été payé, ce qui parait impossible pour un sinistre automobile, il s'agit surement la d'une erreur de saisie, un artefact dans le jeu de donnée qu'il est important de retirer avant l'entrainement des modèles.

On vérifie maintenant s'il n'y a pas des valeurs de ClaimNb qui seraient incohérentes.

In [100]:
print((freq['ClaimNb']).value_counts().sort_index(ascending=False))

ClaimNb
16         1
11         3
9          1
8          1
6          1
5          2
4          7
3         82
2       1784
1      32178
0     643953
Name: count, dtype: int64


La distribution de ClaimNb décroit régulièrement jusqu'à 4, puis devient incohérente : 3 à 11 alors qu'il n'y en a pas plus de 2 de 5 à 9. Une loi de probabilité de comptage n'a pas de telle distribution, il s'agit donc surement d'un artefact.

In [101]:
freq.loc[freq['ClaimNb'] >= 4, ['IDpol', 'ClaimNb', 'Exposure']]


,IDpol,ClaimNb,Exposure
551,1147.0,4,0.08
6539,19471.0,11,1.00
9136,24495.0,4,1.00
25347,54009.0,4,0.56
46470,93954.0,5,1.00
304484,2216294.0,6,0.33
320764,2239279.0,8,0.41
321248,2241683.0,16,0.33
321512,2248174.0,9,0.08
344616,2277762.0,5,0.08


Regarder directement les polices ayant plus un ClaimNb de 4 ou plus le confirme, la pluspart ont des périodes d'exposition courtes : 16 sinistres sur 4 mois, cela dépasse toute réalité de souscription, un tel client serait sans doute résilié.

Nous allons donc ramener toutes ces valeurs à 4, elles représentent 16 polices sur 678 013 soit 0.002%, l'écart que ce changement produira sur la fréquence annualisé du portefeuille est négligeable.

In [102]:
freq['ClaimNb']  = freq['ClaimNb'].clip(upper=4)

J'ai aussi essayé içi d'appliquer un critère de discrimination sur la fréquence implicite des sinistres par police. En effet un raisonnement simple pourrait être que si on a une fréquence implicite très élevé, il s'agit surement d'une anomalie. Or il est, au vu du résultat ci dessous mal adapté car il pénalise les exposition courtes avec peu de sinistres tout en épargnant potentiellement des nombres anormalement grand de sinistre mais sur une longue exposition. 

In [103]:
extremes = freq.loc[freq['ClaimNb'] >= 4].copy()
extremes['freq_implicite'] = extremes['ClaimNb'] / extremes['Exposure']
extremes[['ClaimNb', 'Exposure', 'freq_implicite']].sort_values('freq_implicite', ascending=False)

,ClaimNb,Exposure,freq_implicite
488270,4,0.07,57.142857
551,4,0.08,50.000000
321512,4,0.08,50.000000
344616,4,0.08,50.000000
487268,4,0.08,50.000000
501990,4,0.10,40.000000
364437,4,0.27,14.814815
304484,4,0.33,12.121212
321248,4,0.33,12.121212
320764,4,0.41,9.756098


## Exploration des données

In [104]:
print((freq['ClaimNb']).value_counts().sort_index())

ClaimNb
0    643953
1     32178
2      1784
3        82
4        16
Name: count, dtype: int64


In [105]:
print((sev["ClaimAmount"] <= 1).sum())
print(sev["ClaimAmount"].quantile([0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))
print(sev['ClaimAmount'].value_counts().head(10))

1
0.01       40.0314
0.10      197.3360
0.25      686.8100
0.50     1172.0000
0.75     1228.0800
0.90     2799.0720
0.99    16793.7044
Name: ClaimAmount, dtype: float64
ClaimAmount
1204.00    4792
1128.12    3056
1172.00    2072
1128.00     831
602.00      433
1320.00     175
556.14      155
586.00      104
1500.00     103
564.06       94
Name: count, dtype: int64


on remarque que certaines valeurs reveinnent beaucoup dans les couts d'un sinistre. Surement un montant de provision standard demandé par l'assureur à l'ouverture d'un dossier de sinistre.

Il est très important de le noter car cela aura une incidence sur la suite car ces montants etants vraissemblablement standards et automatiques, il ne dependent pas a premiere vue des caractéristiques de l'assuré, c'est un forfait.

In [106]:
print(freq['ClaimNb'].value_counts().head(10))

ClaimNb
0    643953
1     32178
2      1784
3        82
4        16
Name: count, dtype: int64


In [107]:
print(freq.loc[freq['ClaimNb'] >= 4, ['IDpol', 'ClaimNb', 'Exposure']])

            IDpol  ClaimNb  Exposure
551        1147.0        4      0.08
6539      19471.0        4      1.00
9136      24495.0        4      1.00
25347     54009.0        4      0.56
46470     93954.0        4      1.00
304484  2216294.0        4      0.33
320764  2239279.0        4      0.41
321248  2241683.0        4      0.33
321512  2248174.0        4      0.08
344616  2277762.0        4      0.08
364437  3016883.0        4      0.27
487268  3253234.0        4      0.08
488270  3254353.0        4      0.07
501990  4031494.0        4      0.10
642405  6006882.0        4      0.49
655365  6059824.0        4      0.57


Maintenant que les données ont bien été nettoyées et les valeurs anormales retirée du jeu de données, nous pouvons calculer la fréquence moyenne de sinistre 

In [108]:
print(freq["ClaimNb"].sum() / freq["Exposure"].sum())

0.1006138781922095


Tache 1

la table freq contient les données de contrats (678013) et la table sev celles de sinistres (26639) donc beaucoup plus de sinistres que de contrats (logique)

les deux tables sont relié par un identifiant unique a chaque police : IDpol. Il peut donc y avoir plusieurs sinistres par contrats.

On a une feature Exposure dans la table freq qui correspond à l'exposition en année police.

34060 on eu au moins 1 sinistres ce qui représente 5%, ce qui me parait normal. apres il faut voir sur quelle moyenne de temps de contrat. Sur un moyenne assez longue oui mais sur un courte cela devient suspect.

Le minimum d'exposition est 0.002732 ce qui représente seulement quelque jours, si il y a eu un sinistre cela pose vraiment question

le plus gros sinistre (en couts) est de 4 000 000, il s'agit surement d'un sinistre avec une rente à vie pour l'autre partie.

